# Data Splitting for the project "Fast Nano SLMs for function-calling"

In [7]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [3]:
import warnings
warnings.filterwarnings("ignore")

## GPU (cuda) Info Preview

In [6]:
import torch

if torch.cuda.is_available():
    print("CUDA is available ✅")
    print("Device count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"\n--- GPU {i} ---")
        print("Name:", torch.cuda.get_device_name(i))
        print("Capability:", torch.cuda.get_device_capability(i))
        print("Total memory (GB):", torch.cuda.get_device_properties(i).total_memory / 1e9)
else:
    print("CUDA is NOT available ❌")

CUDA is available ✅
Device count: 1

--- GPU 0 ---
Name: Tesla T4
Capability: (7, 5)
Total memory (GB): 15.637086208


## Dataset Splitting Strategy

In [8]:
import json
from datasets import load_dataset

dataset = load_dataset("Salesforce/xlam-function-calling-60k", split="train")
dataset

README.md: 0.00B [00:00, ?B/s]

xlam_function_calling_60k.json:   0%|          | 0.00/96.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answers', 'tools'],
    num_rows: 60000
})

In [10]:
RANDOM_STATE = 42 # for reproducibility

# Cut 50% of the Dataset: 30,000 samples
subset = dataset.shuffle(seed=42).select(range(30000))

# TEST SET
split_1 = subset.train_test_split(test_size=1500, seed=42)
test_dataset = split_1['test']
remaining_pool = split_1['train']

# VALIDATION SET
split_2 = remaining_pool.train_test_split(test_size=1500, seed=42)
val_dataset = split_2['test']
train_pool = split_2['train']

# TRAIN SFT SET
split_3 = train_pool.train_test_split(train_size=10000, seed=42)
sft_dataset = split_3['train']

# TRAIN GRPO SET
grpo_dataset = split_3['test']

print(f"Test Dataset: {len(test_dataset)} examples")
print(f"Validation Dataset: {len(val_dataset)} examples")
print(f"SFT Dataset: {len(sft_dataset)} examples")
print(f"GRPO Dataset: {len(grpo_dataset)} examples")

Test Dataset: 1500 examples
Validation Dataset: 1500 examples
SFT Dataset: 10000 examples
GRPO Dataset: 17000 examples
